# Seto: SFT (Supervised Fine-Tuning)

Цель: дообучить базовую модель (после pretrain) следовать инструкциям, вести диалог и использовать инструменты (tool call).

Пайплайн:
1. Поиск `final_pretrain.zip` во входных данных `/kaggle/input`
2. Подготовка и нормализация датасетов (UltraChat, WildChat, OASST1, Russian, Hermes tools)
3. Сборка взвешенного микса на 50 000 примеров
4. Обучение SFT (3 000 шагов, Cosine schedule, FP16)
5. Экспорт `final_sft.zip` и тест инференса


In [ ]:
!pip install -q tokenizers datasets


In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
REPO = WORK / "seto"
HF_CACHE = WORK / "hf-sft-cache"
SOURCES = WORK / "seto-sft-sources"
MIX = WORK / "seto-sft-mix"
BASE_MODEL = WORK / "seto-pretrain"
TOKENIZER = WORK / "seto-tokenizer"
OUTPUT = WORK / "seto-sft-v1"

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["HF_HUB_DISABLE_XET"] = "1"

def run(*args, cwd=None):
    print("+", " ".join(map(str, args)), flush=True)
    subprocess.run([str(a) for a in args], cwd=cwd, check=True)

# Поиск pretrain весов в /kaggle/working или /kaggle/input
candidates = []
for p in [WORK / "final_pretrain.zip", WORK / "seto-small-1-full" / "final_pretrain.zip", WORK / "seto-small-1-smoke" / "final_pretrain.zip"]:
    if p.exists() and p.stat().st_size > 1_000_000:
        candidates.append(p)

if not candidates:
    candidates = sorted(Path("/kaggle/input").rglob("final_pretrain.zip"), key=lambda p: p.stat().st_mtime)
if not candidates:
    candidates = sorted(Path("/kaggle/input").rglob("seto_step_*.zip"), key=lambda p: p.stat().st_mtime)
if not candidates:
    candidates = sorted(WORK.rglob("seto_step_*.zip"), key=lambda p: p.stat().st_mtime)

if not candidates:
    raise FileNotFoundError("Не найден чекпойнт pretrain (final_pretrain.zip или seto_step_*.zip) в /kaggle/input или /kaggle/working")

pretrain_zip = candidates[-1]
print(f"Используем pretrain веса: {pretrain_zip} ({pretrain_zip.stat().st_size / 1024**2:.1f} MB)")


In [ ]:
# Клонирование / обновление репозитория
if REPO.exists():
    run("git", "pull", "--ff-only", cwd=REPO)
else:
    run("git", "clone", "https://github.com/mosshaven/seto.git", REPO)

sys.path.insert(0, str(REPO))
print("Репозиторий готов:", REPO)


In [ ]:
# Подготовка данных SFT: скачивание источников и генерация микса
SOURCES.mkdir(parents=True, exist_ok=True)
MIX.mkdir(parents=True, exist_ok=True)

run(
    sys.executable, "scripts/prepare_sft_sources.py",
    "--output-dir", SOURCES,
    "--seed", "42",
    cwd=REPO,
)
run(
    sys.executable, "scripts/prepare_oasst.py",
    "--languages", "ru,en,uk",
    "--max-turns", "12",
    "--max-chars", "8000",
    "--min-reviews", "1",
    "--output", SOURCES / "oasst1.jsonl",
    "--metadata-output", SOURCES / "oasst1.meta.jsonl",
    "--manifest-output", SOURCES / "oasst1.manifest.json",
    cwd=REPO,
)

# Очищаем кэш скачанных сырых файлов для экономии диска Kaggle
shutil.rmtree(HF_CACHE, ignore_errors=True)

run(
    sys.executable, "scripts/mix_sft.py",
    "--config", "configs/sft-v1-first-run.json",
    "--output", MIX / "seto-sft-v1.jsonl",
    "--metadata-output", MIX / "seto-sft-v1.meta.jsonl",
    "--manifest-output", MIX / "seto-sft-v1.manifest.json",
    cwd=REPO,
)

# Распаковка весов и токенизатора из pretrain zip
BASE_MODEL.mkdir(parents=True, exist_ok=True)
TOKENIZER.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(pretrain_zip) as archive:
    names = archive.namelist()
    for filename in ("model.pt", "config.json"):
        matches = [n for n in names if (n == filename or n.endswith("/" + filename)) and "/tokenizer/" not in n]
        if matches:
            with archive.open(matches[0]) as src, (BASE_MODEL / filename).open("wb") as dst:
                shutil.copyfileobj(src, dst)
    for filename in ("tokenizer.json", "config.json"):
        matches = [n for n in names if n == f"tokenizer/{filename}" or n.endswith(f"/tokenizer/{filename}")]
        if matches:
            with archive.open(matches[0]) as src, (TOKENIZER / filename).open("wb") as dst:
                shutil.copyfileobj(src, dst)

# Валидация батча SFT
run(
    sys.executable, "scripts/check_sft_batch.py",
    "--dataset", MIX / "seto-sft-v1.jsonl",
    "--tokenizer", TOKENIZER,
    "--seq-len", "1024",
    cwd=REPO,
)


In [ ]:
# Обучение SFT (Оптимизировано: batch 4, accum 4, без лишнего recompute)
gpu_count = max(1, __import__("torch").cuda.device_count())
train_cmd = [
    "torchrun", "--standalone", f"--nproc_per_node={gpu_count}",
    "scripts/train.py",
    "--stage", "sft",
    "--model-config-file", BASE_MODEL / "config.json",
    "--dataset", MIX / "seto-sft-v1.jsonl",
    "--tokenizer", TOKENIZER,
    "--init-from", BASE_MODEL / "model.pt",
    "--output-dir", OUTPUT,
    "--batch-size", "4",
    "--grad-accum", "4",
    "--seq-len", "1024",
    "--lr", "2e-5",
    "--warmup-steps", "100",
    "--max-steps", "3000",
    "--save-every", "500",
    "--log-every", "10",
    "--fp16",
    "--gradient-checkpointing",
]
print("Команда запуска SFT:")
print(" ".join(map(str, train_cmd)))
run(*train_cmd, cwd=REPO)


In [ ]:
# Проверка экспорта и тест инференса
final_sft = OUTPUT / "final_sft.zip"
root_final_sft = WORK / "final_sft.zip"
if not final_sft.exists():
    raise FileNotFoundError(f"Чекпойнт {final_sft} не найден")

print(f"✅ Успешно создан финальный архив: {final_sft} ({final_sft.stat().st_size / 1024**2:.1f} MB)")

# Копируем в корень Output для удобного скачивания
if final_sft != root_final_sft:
    shutil.copyfile(final_sft, root_final_sft)
    print(f"✅ Архив продублирован в корень Output: {root_final_sft}")

# Быстрый тест диалога
print("\n=== Тестовый ответ Seto-Small ===")
run(
    sys.executable, "scripts/chat.py",
    "--model", str(final_sft),
    "--prompt", "Привет! Расскажи о себе кратко.",
    cwd=REPO,
)
